## OLX


### SCRIPT TEM A FINALIZADE DE PEGAR INFORMAÇÕES DO SITE OLX E VIVAREAL E TABULAR

In [85]:
import requests 
import scrapy
import json
from parsel import Selector
import cloudscraper
import pandas as pd
import time
from datetime import datetime


In [86]:


CONFIG = {
    'AUTOTHROTTLE_ENABLED': True, 
    'MAX_RETRIES': 3,  
    'THROTTLE_DELAY': 3,  
}

def scrape_olx(headers, start_page=1, end_page=101):
    scraper = cloudscraper.create_scraper()
    all_casas = []

    for page in range(start_page, end_page + 1):
        retries = 0
        while retries < CONFIG['MAX_RETRIES']:
            url = f'https://www.olx.com.br/imoveis/estado-mg/belo-horizonte-e-regiao/zona-centro-sul/lourdes?o={page}'
            try:
                r = scraper.get(url, headers=headers)

                if r.status_code == 200:  
                    s = Selector(text=r.text)
                    script_data = s.xpath('//script[@id="__NEXT_DATA__"]/text()').get()

                    if script_data:
                        html = json.loads(script_data)
                        casas = html.get('props', {}).get('pageProps', {}).get('ads', [])
                        all_casas.extend(casas)
                        print(f"✅ Página {page} coletada com sucesso.")
                        break
                    else:
                        print(f"⚠️ Dados não encontrados na página {page}. Tentando novamente...")

                else:
                    print(f"❌ Erro {r.status_code} na página {page}. Tentando novamente...")

            except Exception as e:
                print(f"⚠️ Erro ao coletar a página {page}: {e}. Tentando novamente...")

            retries += 1
            time.sleep(CONFIG['THROTTLE_DELAY'])  

        if retries == CONFIG['MAX_RETRIES']:
            print(f"❌ Falha ao coletar a página {page} após {CONFIG['MAX_RETRIES']} tentativas.")

    return all_casas

def casas_to_dataframe(casas):
    rows = []
    for casa in casas:
        timestamp = casa.get('date')
        formatted_date = datetime.fromtimestamp(timestamp).strftime('%d/%m às %H:%M') if timestamp else None
        row = {
            'Título': casa.get('title'),
            'Preço': casa.get('price'),
            'ID do Anúncio': casa.get('listId'),
            'Localização': casa.get('location'),
            'Município': casa.get('locationDetails', {}).get('municipality'),
            'DDD': casa.get('locationDetails', {}).get('ddd'),
            'Bairro': casa.get('locationDetails', {}).get('neighbourhood'),
            'Estado': casa.get('locationDetails', {}).get('uf'),
            'Categoria': casa.get('category'),
            'Tipo': next((p['value'] for p in casa.get('properties', []) if p['name'] == 'real_estate_type'), None),
            'Condomínio': next((p['value'] for p in casa.get('properties', []) if p['name'] == 'condominio'), None),
            'IPTU': next((p['value'] for p in casa.get('properties', []) if p['name'] == 'iptu'), None),
            'Área Útil': next((p['value'] for p in casa.get('properties', []) if p['name'] == 'size'), None),
            'Quartos': next((p['value'] for p in casa.get('properties', []) if p['name'] == 'rooms'), None),
            'Banheiros': next((p['value'] for p in casa.get('properties', []) if p['name'] == 'bathrooms'), None),
            'Vagas na Garagem': next((p['value'] for p in casa.get('properties', []) if p['name'] == 'garage_spaces'), None),
            'Data Anúncio': formatted_date  
        }
        rows.append(row)
        print(f"✅ Anúncio coletado: {row['Título']} | Preço: {row['Preço']}")
    return pd.DataFrame(rows)

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'
}

casas = scrape_olx(headers, start_page=1, end_page=100)
df_casas = casas_to_dataframe(casas)
#df_casas.to_('casas_olx.xlsx')


✅ Página 1 coletada com sucesso.
✅ Página 2 coletada com sucesso.
✅ Página 3 coletada com sucesso.
✅ Página 4 coletada com sucesso.
✅ Página 5 coletada com sucesso.
✅ Página 6 coletada com sucesso.
✅ Página 7 coletada com sucesso.
✅ Página 8 coletada com sucesso.
✅ Página 9 coletada com sucesso.
✅ Página 10 coletada com sucesso.
✅ Página 11 coletada com sucesso.
✅ Página 12 coletada com sucesso.
✅ Página 13 coletada com sucesso.
✅ Página 14 coletada com sucesso.
✅ Página 15 coletada com sucesso.
✅ Página 16 coletada com sucesso.
✅ Página 17 coletada com sucesso.
✅ Página 18 coletada com sucesso.
✅ Página 19 coletada com sucesso.
✅ Página 20 coletada com sucesso.
✅ Página 21 coletada com sucesso.
✅ Página 22 coletada com sucesso.
✅ Página 23 coletada com sucesso.
✅ Página 24 coletada com sucesso.
✅ Página 25 coletada com sucesso.
✅ Página 26 coletada com sucesso.
✅ Página 27 coletada com sucesso.
✅ Página 28 coletada com sucesso.
✅ Página 29 coletada com sucesso.
✅ Página 30 coletada co

In [87]:
df_casas

""


In [88]:


def casas_to_dataframe_from_df(df_casas):
    empreendimentos = []
    tipologias = []
    unidades = []

    for _, casa in df_casas.iterrows():

        formatted_date = casa.get('Data Anúncio')


        empreendimento_nome = casa.get('Bairro', 'Desconhecido')
        empreendimento_endereco = casa.get('Localização', 'Desconhecido')
        empreendimento_id = f"{empreendimento_nome}_{empreendimento_endereco}"


        quartos = casa.get('Quartos')
        banheiros = casa.get('Banheiros')
        tipologia_id = f"{empreendimento_id}_Q{quartos}_B{banheiros}"

        empreendimentos.append({
            'Empreendimento': empreendimento_id,
            'ID do Anúncio': casa.get('ID do Anúncio'),
            'Nome': empreendimento_nome,
            'Endereço': empreendimento_endereco,
            'Bairro': casa.get('Bairro'),
            'Município': casa.get('Município'),
            'Estado': casa.get('Estado'),
        })


        tipologias.append({
            'Tipologia': tipologia_id,
            'ID do Anúncio': casa.get('ID do Anúncio'),
            'Empreendimento': empreendimento_id,
            'Número de Quartos': quartos,
            'Número de Banheiros': banheiros,
            'Área Útil': casa.get('Área Útil'),
            'Vagas na Garagem': casa.get('Vagas na Garagem'),
            'Preço Médio': casa.get('Preço'),
        })

        # Unidades
        unidades.append({
            'Tipologia': tipologia_id,
            'ID do Anúncio': casa.get('ID do Anúncio'),
            'Título': casa.get('Título'),
            'Preço': casa.get('Preço'),
            'Condomínio': casa.get('Condomínio'),
            'IPTU': casa.get('IPTU'),
            'Data do Anúncio': formatted_date,
        })

    return (
        pd.DataFrame(empreendimentos).drop_duplicates(),
        pd.DataFrame(tipologias).drop_duplicates(),
        pd.DataFrame(unidades).drop_duplicates()
    )

def salvar_dados(empreendimentos, tipologias, unidades):
    data_atual = datetime.now().strftime("%Y-%m")

    empreendimentos.to_csv(f"empreendimentos_{data_atual}.csv", index=False, encoding='utf-8')
    tipologias.to_csv(f"tipologias_{data_atual}.csv", index=False, encoding='utf-8')
    unidades.to_csv(f"unidades_{data_atual}.csv", index=False, encoding='utf-8')

    print(f"📁 Dados salvos com sucesso para {data_atual}!")

df_empreendimentos, df_tipologias, df_unidades = casas_to_dataframe_from_df(df_casas)

salvar_dados(df_empreendimentos, df_tipologias, df_unidades)


📁 Dados salvos com sucesso para 2025-01!


In [89]:
df_empreendimentos
#df_casas[0]

""


In [90]:
df_tipologias

""


In [91]:
df_unidades

""


In [92]:
total_empreendimentos = df_empreendimentos.shape[0]
print(f"Quantidade total de empreendimentos identificados OLX: {total_empreendimentos}")

Quantidade total de empreendimentos identificados OLX: 0


In [93]:
df_tipologias.columns

RangeIndex(start=0, stop=0, step=1)

In [94]:
# tipologia olxl

tipologias_frequentes = df_tipologias['Número de Quartos'].value_counts()



tipologias_frequentes = tipologias_frequentes.reset_index()
tipologias_frequentes.columns = ['Quantidade quartos', 'Quantidade Imoveis']
#tipologias_frequentes = pd.DataFrame(tipologias_frequentes).reset_index()
tipologias_frequentes

KeyError: 'Número de Quartos'

In [ ]:



# faixa preco olx
df_unidades['Preço'] = df_unidades['Preço'].replace('[^0-9]', '', regex=True).astype(float)

df_tipologias['Área Útil'] = df_tipologias['Área Útil'].replace('[^0-9]', '', regex=True).astype(float)
df_unidades['Preço por m²'] = df_unidades['Preço'] / df_tipologias['Área Útil']
df_unidades
faixa_preco = df_unidades.groupby(pd.cut(df_unidades['Preço por m²'], bins=[0, 5000, 10000, 15000, 20000, 25000])).size()
faixa_preco_df = faixa_preco.reset_index()
faixa_preco_df.columns = ['Faixa de Preço (R$) por m²', 'Quantidade Imoveis']

faixa_preco = pd.DataFrame(faixa_preco_df)
faixa_preco


# OLX DUPLICADOS

In [ ]:

if isinstance(df_casas, list):
    df_casas = pd.DataFrame(df_casas)

print("Colunas disponíveis no DataFrame:")


duplicated = df_casas.duplicated(subset=['Categoria', 'IPTU', 'Área Útil', 'Quartos', 'Banheiros', 'Vagas na Garagem'], keep=False)
duplicated
duplicated_rows = df_casas[duplicated]
duplicated_rows[duplicated_rows['Título']=='CARNAVAL - BH - 2025 - 30 HÓSPEDES']

duplicated_rows_sorted = duplicated_rows.sort_values(by='Título', ascending=True)
duplicated_rows_sorted




# VIVALREAL

In [ ]:
import cloudscraper
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

def scrape_vivareal(headers, start_page=1, end_page=50):
    scraper = cloudscraper.create_scraper()
    all_imoveis = []

    for page in range(start_page, end_page + 1):
        print(f"Coletando dados da página {page}...")
        time.sleep(1)
        url = f"https://www.vivareal.com.br/venda/minas-gerais/belo-horizonte/bairros/lourdes/?pagina={page}#onde=Brasil,Minas%20Gerais,Belo%20Horizonte,Bairros,Lourdes,,,,BR%3EMinas%20Gerais%3ENULL%3EBelo%20Horizonte%3EBarrios%3ELourdes,,,"

        response = scraper.get(url, headers=headers)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, 'html.parser')

            property_cards = soup.find_all('article', class_='property-card__container')

            for card in property_cards:
                titulo = card.find('span', class_='property-card__title').get_text(strip=True) if card.find('span', class_='property-card__title') else None
                endereco = card.find('span', class_='property-card__address').get_text(strip=True) if card.find('span', class_='property-card__address') else None
                
                preco_div = card.find('div', class_='property-card__price js-property-card-prices js-property-card__price-small')
                preco = preco_div.find('p').get_text(strip=True) if preco_div and preco_div.find('p') else None

                area = card.find('span', class_='property-card__detail-value js-property-card-detail-area').get_text(strip=True) if card.find('span', class_='property-card__detail-value js-property-card-detail-area') else None
                quartos = card.find('li', class_='property-card__detail-room')
                quartos = quartos.find('span', class_='property-card__detail-value').get_text(strip=True) if quartos else None
                banheiros = card.find('li', class_='property-card__detail-bathroom')
                banheiros = banheiros.find('span', class_='property-card__detail-value').get_text(strip=True) if banheiros else None
                vagas = card.find('li', class_='property-card__detail-garage')
                vagas = vagas.find('span', class_='property-card__detail-value').get_text(strip=True) if vagas else None

                all_imoveis.append({
                    'Título': titulo,
                    'Endereço': endereco,
                    'Preço': preco,
                    'Área': area,
                    'Quartos': quartos,
                    'Banheiros': banheiros,
                    'Vagas': vagas,
                    'Página': page  
                })

        else:
            print(f" Erro ao acessar a página {page}. Código HTTP: {response.status_code}")

    df_imoveis = pd.DataFrame(all_imoveis)
    print("Coleta finalizada. Total de imóveis coletados:", len(df_imoveis))

    df_imoveis.to_csv('imoveis_vivareal.csv', index=False, encoding='utf-8')
    return df_imoveis

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
    'Cache-Control': 'no-cache'
}


df_imoveis = scrape_vivareal(headers, start_page=1, end_page=50)
residencial = [
    "Apartamento", "Casa", "Casa de Condomínio", "Cobertura", "Flat",
    "Kitnet/Conjugado", "Lote/Terreno", "Sobrado", "Edifício Residencial",
    "Fazenda/Sítios/Chácaras"
]

comercial = [
    "Consultório", "Galpão/Depósito/Armazém", "Imóvel Comercial",
    "Lote/Terreno", "Ponto Comercial/Loja/Box", "Sala/Conjunto",
    "Prédio/Edifício Inteiro"
]

def categorizar_titulo(titulo, categorias):
    for categoria in categorias:
        if categoria in titulo:
            return categoria
    return None

df_imoveis['Residencial'] = df_imoveis['Título'].apply(lambda x: categorizar_titulo(x, residencial))
df_imoveis['Comercial'] = df_imoveis['Título'].apply(lambda x: categorizar_titulo(x, comercial))


def extrair_metros(titulo):
    match = re.search(r'\d+\s*m²', titulo)
    if match:
        return match.group().strip()
    return None

df_imoveis['Metros'] = df_imoveis['Título'].apply(extrair_metros)


In [ ]:
df_imoveis

In [ ]:
from datetime import datetime
import pandas as pd

def imoveis_to_dataframe(df_imoveis):
    empreendimentos = []
    tipologias = []
    unidades = []

    for _, row in df_imoveis.iterrows():
        empreendimento_id = f"{row['Endereço']}_{row['Página']}"
        tipologia_id = f"{empreendimento_id}_Q{row['Quartos']}_B{row['Banheiros']}"

        empreendimentos.append({
            'Empreendimento': empreendimento_id,
            'Endereço': row['Endereço'],
            'Página': row['Página'],
            'Tipo': 'Residencial' if pd.notna(row['Residencial']) else 'Comercial',
            'Categoria': row['Residencial'] if pd.notna(row['Residencial']) else row['Comercial']
        })

        tipologias.append({
            'Tipologia': tipologia_id,
            'Empreendimento': empreendimento_id,
            'Número de Quartos': row['Quartos'],
            'Número de Banheiros': row['Banheiros'],
            'Área (m²)': row['Metros'],
            'Preço': row['Preço'],
            'Vagas na Garagem': row['Vagas']
        })

        unidades.append({
            'Tipologia': tipologia_id,
            'Título': row['Título'],
            'Preço': row['Preço'],
            'Área (m²)': row['Metros'],
            'Página': row['Página']
        })

    return (
        pd.DataFrame(empreendimentos).drop_duplicates(),
        pd.DataFrame(tipologias).drop_duplicates(),
        pd.DataFrame(unidades).drop_duplicates()
    )

df_empreendimentos, df_tipologias, df_unidades = imoveis_to_dataframe(df_imoveis)
df_empreendimentos




In [ ]:
df_empreendimentos

In [ ]:
df_tipologias

In [ ]:
df_unidades

In [ ]:

total_empreendimentos_vivareal = df_imoveis.shape[0]
print(f"Quantidade total de empreendimentos identificados VIVAREAL: {total_empreendimentos_vivareal}")




In [ ]:
#tipologia vivareal
tipologias_frequentes = df_imoveis['Quartos'].value_counts()
tipologias_frequentes_df = tipologias_frequentes.reset_index()
tipologias_frequentes_df.columns = ['Quantidade de Quartos', 'Quantidade de Imóveis']
tipologias_frequentes_df



In [ ]:
#faixa preco vivareal

df_imoveis['Preço'] = df_imoveis['Preço'].replace('[^0-9]', '', regex=True).astype(float)
df_imoveis['Metros'] = df_imoveis['Metros'].replace('[^0-9]', '', regex=True).astype(float)

df_imoveis['Preço por m²'] = df_imoveis['Preço'] / df_imoveis['Metros']

faixa_preco = df_imoveis.groupby(pd.cut(df_imoveis['Preço por m²'], bins=[0, 5000, 10000, 15000, 20000, 25000])).size()
faixa_preco_df = faixa_preco.reset_index()
faixa_preco_df.columns = ['Faixa de Preço (R$) por m²', 'Quantidade de Imóveis']
faixa_preco_df

# GRAFICOS

In [ ]:
import plotly.express as px
import pandas as pd

dados_comparacao = pd.DataFrame({
    'Fonte': ['Viva Real', 'OLX'],
    'Quantidade de Empreendimentos': [total_empreendimentos_vivareal, total_empreendimentos]
})

fig_comparacao = px.bar(
    dados_comparacao,
    x='Fonte',
    y='Quantidade de Empreendimentos',
    title='Comparação da Quantidade Total de Empreendimentos',
    labels={'Fonte': 'Fonte de Dados', 'Quantidade de Empreendimentos': 'Total de Empreendimentos'},
    text='Quantidade de Empreendimentos',
    color='Fonte'
)

fig_comparacao.show()


In [ ]:
# Convertendo os intervalos para string para evitar erro de JSON no Plotly
faixa_preco_df['Faixa de Preço (R$) por m²'] = faixa_preco_df['Faixa de Preço (R$) por m²'].astype(str)

# Criar gráfico da distribuição de preços por m²
fig_faixa_preco = px.bar(
    faixa_preco_df,
    x='Faixa de Preço (R$) por m²',
    y='Quantidade de Imóveis',
    title='OLX - Distribuição de Imóveis por Faixa de Preço por m²',
    labels={'Faixa de Preço (R$) por m²': 'Faixa de Preço (R$) por m²', 'Quantidade de Imóveis': 'Quantidade'},
    text='Quantidade de Imóveis'
)

# Exibir o gráfico
fig_faixa_preco.show()




In [ ]:
# --- Tipologias Mais Frequentes ---

# Contar a frequência dos números de quartos
tipologias_frequentes = df_tipologias['Número de Quartos'].value_counts().reset_index()
tipologias_frequentes.columns = ['Quantidade de Quartos', 'Quantidade de Imóveis']

# Criar gráfico da distribuição de tipologias
fig_tipologias = px.bar(
    tipologias_frequentes,
    x='Quantidade de Quartos',
    y='Quantidade de Imóveis',
    title='OLX - Distribuição de Imóveis por Quantidade de Quartos',
    labels={'Quantidade de Quartos': 'Número de Quartos', 'Quantidade de Imóveis': 'Quantidade'},
    text='Quantidade de Imóveis'
)

# Exibir o gráfico
fig_tipologias.show()

In [ ]:



# Criando os gráficos
fig_tipologias = px.bar(
    tipologias_frequentes_df,
    x='Quantidade de Quartos',
    y='Quantidade de Imóveis',
    title='VIVALREAL - Distribuição de Tipologias por Quartos',
    labels={'Quantidade de Quartos': 'Quantidade de Quartos', 'Quantidade de Imóveis': 'Quantidade de Imóveis'},
    text='Quantidade de Imóveis'
)


faixa_preco_df['Faixa de Preço (R$) por m²'] = faixa_preco_df['Faixa de Preço (R$) por m²'].astype(str)

fig_faixa_preco = px.bar(
    faixa_preco_df,
    x='Faixa de Preço (R$) por m²',
    y='Quantidade de Imóveis',
    title='VIVALREAL - Distribuição de Imóveis por Faixa de Preço por m²',
    labels={'Faixa de Preço (R$) por m²': 'Faixa de Preço (R$) por m²', 'Quantidade de Imóveis': 'Quantidade'},
    text='Quantidade de Imóveis'
)





fig_faixa_preco.show()


In [ ]:
fig_tipologias.show()